# Replicating Baker et al 2022 network analyses on updated Baker auditory connectome data only

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy import stats
import sys

### Compiling data

In [7]:
baker_data = pd.read_csv('baker_neurons2026.csv')
baker_data

,Neuron name (Baker et al 2022),Cell type (Baker et al 2022),Neuron name (Lockwood et al),Cell type (Lockwood et al),Coordinates,Codex June 2024,HB type
0,A1_L1,A1,A1_R1,A1,[160548 55409 4977],720575940631693610,AMMC-A1
1,A1_L2,A1,A1_R2,A1,[161229 56723 4921],720575940631200327,AMMC-A1
2,A1_L3,A1,A1_R3,A1,[159857 56701 5018],720575940618522433,AMMC-A1
3,A1_R1,A1,A1_L1,A1,[105496 54286 5405],720575940612001489,AMMC-A1
4,A1_R2,A1,A1_L2,A1,[105654 55604 5474],720575940616313942,AMMC-A1
...,...,...,...,...,...,...,...
895,NaN,NaN,JO-B_L162,JO-B,NaN,720575940631670584,NaN
896,NaN,NaN,JO-B_L163,JO-B,NaN,720575940617244049,NaN
897,NaN,NaN,JO-B_L164,JO-B,NaN,720575940617407929,NaN
898,NaN,NaN,JO-B_L165,JO-B,NaN,720575940620257216,NaN


In [15]:
connections = pd.read_csv('connections_princeton.csv.gz')
connections = connections.groupby(['pre_root_id', 
                                   'post_root_id', 
                                   'nt_type'], as_index=False)[['syn_count']].sum().sort_values(by = 'syn_count', 
                                                                                                ascending=False).drop_duplicates(subset=['pre_root_id', 
                                                                                                                                         'post_root_id'], keep='first')
connections

,pre_root_id,post_root_id,nt_type,syn_count
3687238,720575940650935673,720575940640736371,GABA,2633
2320879,720575940628069501,720575940633305681,GABA,2597
3596036,720575940644702112,720575940625952755,GABA,2238
1631524,720575940623475324,720575940627497244,GABA,2179
1246516,720575940620975696,720575940624547622,DA,1872
...,...,...,...,...
3073466,720575940633914413,720575940615905130,GLUT,5
1582223,720575940623166600,720575940622950424,ACH,5
3073469,720575940633914413,720575940622525168,GLUT,5
1582220,720575940623166600,720575940618583877,ACH,5


In [16]:
aud_ids = baker_data['Codex June 2024']
conns = connections.copy(deep=True)
conns = conns[(conns['pre_root_id'].isin(aud_ids)) | (conns['post_root_id'].isin(aud_ids))]
conns

,pre_root_id,post_root_id,nt_type,syn_count
1910866,720575940625441003,720575940623113752,GABA,879
2840264,720575940631698340,720575940629513478,GABA,858
81220,720575940606354633,720575940623113752,GABA,599
2110532,720575940626848275,720575940629513478,GABA,553
2539114,720575940629513478,720575940626848275,DA,492
...,...,...,...,...
1577861,720575940623146378,720575940614310802,ACH,5
1577859,720575940623146378,720575940610638434,ACH,5
3074825,720575940633927519,720575940627642818,GLUT,5
1585494,720575940623189799,720575940622123981,ACH,5


In [36]:
#include types

firstmerge = conns.merge(
    baker_data,
    how = 'left',
    left_on = 'pre_root_id',
    right_on = 'Codex June 2024'
).drop(columns = ['Neuron name (Baker et al 2022)', 
                  'Neuron name (Lockwood et al)', 
                  'Cell type (Lockwood et al)',
                  'Coordinates ',
                  'Codex June 2024',
                  'HB type']).rename(columns={'Cell type (Baker et al 2022)':'type_pre'})

conns_aud = firstmerge.merge(
    baker_data,
    how = 'left',
    left_on = 'post_root_id',
    right_on = 'Codex June 2024'
).drop(columns = ['Neuron name (Baker et al 2022)', 
                  'Neuron name (Lockwood et al)', 
                  'Cell type (Lockwood et al)',
                  'Coordinates ',
                  'Codex June 2024',
                  'HB type']).rename(columns={'Cell type (Baker et al 2022)':'type_post'})

conns_aud


,pre_root_id,post_root_id,nt_type,syn_count,type_pre,type_post
0,720575940625441003,720575940623113752,GABA,879,NaN,AVLP_pr04
1,720575940631698340,720575940629513478,GABA,858,NaN,AVLP_pr04
2,720575940606354633,720575940623113752,GABA,599,NaN,AVLP_pr04
3,720575940626848275,720575940629513478,GABA,553,NaN,AVLP_pr04
4,720575940629513478,720575940626848275,DA,492,AVLP_pr04,NaN
...,...,...,...,...,...,...
60101,720575940623146378,720575940614310802,ACH,5,NaN,B1-4
60102,720575940623146378,720575940610638434,ACH,5,NaN,put_B1
60103,720575940633927519,720575940627642818,GLUT,5,NaN,AVLP_pr01
60104,720575940623189799,720575940622123981,ACH,5,NaN,WV-WV-1


NaN = non-auditory (according to Baker et al 2022 dataset)

### Analysis

In [37]:
#turn into graph

G_aud = nx.DiGraph()

for i, row in conns_aud.iterrows():
    G_aud.add_node(row['pre_root_id'], label=['type_pre'])
    G_aud.add_node(row['post_root_id'], label=['type_post'])

for i, row in conns_aud.iterrows():
    G_aud.add_edge(row['pre_root_id'], row['post_root_id'], weight=row['syn_count'])

In [38]:
#add analysis functions from Baker et al (2022)

def get_gc(g):
    """Return graph condensation."""
    scc = list(nx.strongly_connected_components(g))
    gc = nx.algorithms.components.condensation(g, scc=scc)
    return gc, scc

def calc_treeness(gc):
    """Return treeness."""
    assert nx.is_directed_acyclic_graph(gc)
    
    k_ins = gc.in_degree()
    k_outs = gc.out_degree()
    
    max_nodes = np.array([v for v, k_in in k_ins if k_in == 0])
    min_nodes = np.array([v for v, k_out in k_outs if k_out == 0])
    
    if len(max_nodes) == len(min_nodes) == len(gc):
        return 0., 0., 0.
   
    h_fwds = []
    for v in max_nodes:
        # count all paths from v to each min_node
        paths = sum([list(nx.all_simple_paths(gc, v, u)) for u in min_nodes if (u != v and nx.has_path(gc, v, u))], [])
        
        if len(paths) == 0:
            h_fwds.append(0)  # only one path from v (to self), so 0 entropy
            continue
            
        p_paths = np.array([np.prod([1/k_outs[v_] for v_ in path[:-1]]) for path in paths])  # prob of path is product of all out_degrees along path
        
        np.testing.assert_almost_equal(np.sum(p_paths), 1)
        p_paths /= np.sum(p_paths)
        
        h_fwds.append(stats.entropy(p_paths, base=2))
        
    h_fwd = np.sum(h_fwds)/len(max_nodes)
    
    h_baks = []
    for u in min_nodes:
        # count all paths to u from each max_node
        paths = sum([list(nx.all_simple_paths(gc, v, u)) for v in max_nodes if (u != v and nx.has_path(gc, v, u))], [])
        
        if len(paths) == 0:
            h_fwds.append(0)  # only one path to u (from self), so 0 entropy
            continue
    
        p_paths = np.array([np.prod([1/k_ins[u_] for u_ in path[1:]]) for path in paths])  # prob of path is product of all in_degrees along path
        
        np.testing.assert_almost_equal(np.sum(p_paths), 1)
        p_paths /= np.sum(p_paths)
        
        h_baks.append(stats.entropy(p_paths, base=2))
        
    h_bak = np.sum(h_baks)/len(min_nodes)
    
    return (h_fwd - h_bak) / max(h_fwd, h_bak), h_fwd, h_bak


def calc_feedforwardness(gc):
    """Return feedforwardness"""
    k_ins = gc.in_degree()
    k_outs = gc.out_degree()
    
    max_nodes = np.array([v for v, k_in in k_ins if k_in == 0])
    min_nodes = np.array([v for v, k_out in k_outs if k_out == 0])
    
    if len(max_nodes) == len(min_nodes) == len(gc):
        return 0.
    
    alph = {i: len(gc.nodes[i]['members']) for i in gc.nodes()}
    
    paths_all = []
    p_paths_all = []

    for v in max_nodes:
#         paths = sum([list(nx.all_simple_paths(gc, v, u)) for u in min_nodes if nx.has_path(gc, v, u)], [])
        paths = sum([list(nx.all_simple_paths(gc, v, u)) for u in min_nodes if (u != v and nx.has_path(gc, v, u))], [])
        
        if len(paths) == 0:
            continue
            
        p_paths = [np.prod([1/k_outs[v_] for v_ in path[:-1]]) for path in paths]
        
        np.testing.assert_almost_equal(np.sum(p_paths), 1)
        p_paths /= np.sum(p_paths)
        
        paths_all.extend(paths)
        p_paths_all.extend(p_paths)
        
    p_paths_all = np.array(p_paths_all) / len(max_nodes)
    
    ffs = []
    
    for path in paths_all:
        
        ffs.append(len(path)/np.sum([alph[i] for i in path]))
    
    return np.mean(ffs)

def calc_orderability(gc):
    """Return orderability."""
    k_ins = gc.in_degree()
    k_outs = gc.out_degree()
    
    max_nodes = np.array([v for v, k_in in k_ins if k_in == 0])
    min_nodes = np.array([v for v, k_out in k_outs if k_out == 0])
       
    alph = np.array([len(gc.nodes[v]['members']) for v in gc.nodes() if (k_ins[v] or k_outs[v])])
    
    return np.sum(alph == 1)/alph.sum()

In [39]:
#get graph condensation

gc_aud_test, scc_aud_test = get_gc(G_aud)

In [ ]:
t_aud_test = calc_treeness(gc_aud_test)[0] #treeness
f_aud_test = calc_feedforwardness(gc_aud_test) #feedforwardness
o_aud_test = calc_orderability(gc_aud_test) #orderability

t_aud_test, f_aud_test, o_aud_test

In [ ]:
min_syns = np.arange(15) + 1

ts_aud = np.nan * np.zeros(len(min_syns))
fs_aud = np.nan * np.zeros(len(min_syns))
os_aud = np.nan * np.zeros(len(min_syns))
alphs_all = []

results = []

for i, min_syn in enumerate(min_syns):
    sys.stdout.write('.')
    in_threshold = conns_aud[conns_aud['syn_count'] >= min_syn]

    if in_threshold.empty:
        ts_aud[i] = np.nan
        fs_aud[i] = np.nan
        os_aud[i] = np.nan
        alphs_all.append([])
        results.append({
            'min_syn': min_syn,
            'treeness': np.nan,
            'feedforwardness': np.nan,
            'orderability': np.nan,
            'num_scc': 0
        })
        continue

    g_aud = nx.from_pandas_edgelist(
        in_threshold,
        source = 'pre_root_id',
        target = 'post_root_id',
        edge_attr = 'syn_count',
        create_using = nx.DiGraph
    )

    gc_aud, scc_aud = get_gc(g_aud)
    
    t_aud = calc_treeness(gc_aud)[0]
    f_aud = calc_feedforwardness(gc_aud)
    o_aud = calc_orderability(gc_aud)

    ts_aud[i] = t_aud
    fs_aud[i] = f_aud
    os_aud[i] = o_aud
    
    alphs_all.append([len(scc) for scc in scc_aud])

    results.append({
        'min_syn': min_syn,
        'treeness': t_aud,
        'feedforwardness': f_aud,
        'orderability': o_aud,
        'num_scc': len(scc_aud)
        })
    
    df_tfo_aud = pd.DataFrame(results)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 6), tight_layout=True)

axs[0].plot(df_tfo_aud['min_syn'], df_tfo_aud['treeness'], lw=3, label='Treeness')
axs[0].plot(df_tfo_aud['min_syn'], df_tfo_aud['feedforwardness'], lw=3, label='Feedforwardness')
axs[0].plot(df_tfo_aud['min_syn'], df_tfo_aud['orderability'], lw=3, label='Orderability')

axs[0].set_xlabel('Min # synapses per connection')
axs[0].set_ylabel('T, F, O')
axs[0].legend(ncol=3)
axs[0].grid(True)

axs[1].plot(df_tfo_aud['min_syn'], df_tfo_aud['num_scc'], lw=3, color='black')

axs[1].set_xlabel('Min # synapses per connection')
axs[1].set_ylabel('# SCCs')
axs[1].grid(True)


plt.show()

In [ ]:
min_syns_rand = [1, 5, 10]

for cs, min_syn in enumerate(min_syns_rand):
    
    g_fly = nx.DiGraph()
    
    for i, row in conns_aud.iterrows():
        if row['syn_count'] >= min_syn:
            g_fly.add_node(row['pre_root_id'], label=['type_pre'])
            g_fly.add_node(row['post_root_id'], label=['type_post'])
            g_fly.add_edge(row['pre_root_id'], row['post_root_id'], weight=row['syn_count'])

    gc_fly, scc_fly = get_gc(g_fly)
    
    # get graphical attributes of nodes
    assert nx.is_directed_acyclic_graph(gc_fly)
    
    node_list = np.array(list(gc_fly.nodes()))
    
    k_ins = gc_fly.in_degree()
    k_outs = gc_fly.out_degree()
    
    max_nodes = np.array([v for v, k_in in k_ins if k_in == 0])
    
    xs = {v: 0 for v in max_nodes}
    member_dict = {v: gc_fly.nodes()[v]['members'] for v in gc_fly.nodes()}
    
    prev_nodes = max_nodes.copy()
    
    for i in range(100):
        # get all children of previous set of nodes
        child_list = sum([list(gc_fly.successors(v)) for v in prev_nodes], [])
        
        if not child_list:
            break
        next_nodes = np.array(list(set(child_list)))
        
        for v in next_nodes:
            xs[v] = i+1
            
        prev_nodes = next_nodes
    else:
        raise Exception('for loop reached 100 iters')
    
    for v in node_list:
        if k_ins[v] == k_outs[v] == 0:
            xs[v] = np.nan
            
    x_max = np.nanmax(list(xs.values()))
    for v in node_list:
        if k_outs[v] == 0 and k_ins[v] > 0:
            xs[v] = x_max
         
    # set y positions
    ys = {v: np.nan for v in node_list}
    for x_ in xs.values():
        nodes_this_x = [v for v in gc_fly.nodes() if xs[v] == x_]
        ys_ = np.random.permutation(len(nodes_this_x))
        for y_, v_ in zip(ys_, nodes_this_x):
            ys[v_] = y_ - len(nodes_this_x)/2
            
    pos = {v: (xs[v], ys[v]) for v in node_list}
    alphs = np.array([len(member_dict[v]) for v in node_list])
    
    unconnected = [v for v in node_list if np.isnan(pos[v][0])]
    n_unconnected = len(unconnected)
    
    xr = np.nanmax(list(xs.values())) - np.nanmin(list(xs.values()))
    x_unc = np.nanmin(list(xs.values())) + xr/50 * (np.arange(n_unconnected)%50)
    y_unc = np.nanmax(list(ys.values()))*(1.25 + .05*(np.arange(n_unconnected) // 50))
    
    for cv, v in enumerate(unconnected):
        pos[v] = (x_unc[cv], y_unc[cv])
    
    fig, ax = plt.subplots(1, 1, figsize=(13, 8.5), tight_layout=True)
    nx.drawing.draw_networkx(gc_fly, pos=pos, node_size=10, ax=ax, with_labels=False, arrows=False, arrowsize=10)
    ax.scatter(*zip(*[pos[v] for v in node_list]), s=25*alphs, zorder=1000, c='limegreen')
    ax.set_title(f'At least {min_syn} synapses per connection ({n_unconnected} unconnected nodes)')
        
print(f'SCC sizes (min_syn = {min_syn}):')
print(sorted(alphs)[::-1])
print(f'sum = {np.sum(alphs)}\n')